In [1]:
%pip install folium

Note: you may need to restart the kernel to use updated packages.


In [2]:
import csv
import folium
 
csv_file = "2.5.csv"
 
with open(csv_file, encoding="utf-8-sig") as file:
    earthquakes = list(csv.DictReader(file))
 
print(f"{len(earthquakes)}件のデータを読みました")

1854件のデータを読みました


#Notebook
最新のデータを活用しました。

In [3]:
earthquake_map = folium.Map(location=[20, 0], zoom_start=2)
 
for quake in earthquakes:
    folium.Marker(
        location=[float(quake["latitude"]), float(quake["longitude"])],
        tooltip=f"M {quake['mag']}"
    ).add_to(earthquake_map)
 
earthquake_map

In [7]:
#上の地図の改良
import folium
from folium import Element
import json

# 1. 地図の作成（高さや表示設定を調整）
earthquake_map = folium.Map(
    location=[36, 138], 
    zoom_start=5, 
    tiles="OpenStreetMap",
    width="100%", 
    height="520px"
)

total_count = 0
max_mag = 0.0
max_quake_info = ""
max_lat = 0
max_lon = 0
quakes_data = []

for quake in earthquakes:
    if not quake.get("mag"):
        continue
    
    total_count += 1
    lat = float(quake["latitude"])
    lon = float(quake["longitude"])
    mag = float(quake["mag"])
    
    # 2. 日時を見やすく整形（TやZを取り除く、またはスライスする）
    raw_time = quake.get("time", "不明")
    if "T" in raw_time:
        date_part, time_part = raw_time.split("T")
        time_str = f"{date_part} {time_part[:5]}" # 例: 2026-10-14 12:34
    else:
        time_str = raw_time

    depth = quake.get("depth", "不明")
    # 地名データがあれば取得、なければ緯度経度にする
    place = quake.get("place", "").strip()
    if not place:
        place = f"緯度: {lat}, 経度: {lon}"

    # 最大Mの更新
    if mag > max_mag:
        max_mag = mag
        max_quake_info = f"{place} (M{mag})"
        max_lat = lat
        max_lon = lon

    # マグニチュードに応じた色分け
    if mag >= 5.0:
        color = "red"
    elif mag >= 3.0:
        color = "orange"
    else:
        color = "green"

    quakes_data.append({
        "lat": lat,
        "lon": lon,
        "mag": mag,
        "time": time_str,
        "depth": depth,
        "place": place,
        "color": color
    })

# 3. 右下のサマリーボックス（余白やレイアウトを最適化）
interactive_legend_html = f"""
<div id="quake-info-box" style="
    position: absolute; 
    bottom: 20px; 
    right: 20px; 
    width: 280px; 
    background-color: white; 
    border: 2px solid #555; 
    z-index: 9999; 
    font-size: 13px; 
    padding: 12px; 
    border-radius: 8px; 
    box-shadow: 0 4px 8px rgba(0,0,0,0.3);
    font-family: sans-serif;">
    <p style="margin: 0 0 6px 0; font-weight: bold; color: #333;">📊 地震データサマリー</p>
    <hr style="margin: 4px 0 8px 0; border: none; border-top: 1px solid #ddd;">
    <div id="info-content">
        <p style="margin: 4px 0; color: #666; font-size: 12px;">📌 マップ上のピンをクリックすると、ここに詳細が表示されます。</p>
        <hr style="margin: 8px 0; border: none; border-top: 1px dashed #eee;">
        <p style="margin: 4px 0;">総観測数: <b>{total_count}</b> 件</p>
        <p style="margin: 4px 0;">最大規模: <b style="color: red;">M {max_mag}</b></p>
        <p style="margin: 4px 0; font-size: 11px; color: #333;"><b>最大発生地:</b><br>{max_quake_info}</p>
    </div>
    <div style="margin-top: 8px; font-size: 11px; border-top: 1px dashed #eee; padding-top: 4px; display: flex; justify-content: space-between;">
        <span><span style="color: red; font-weight: bold;">■</span> M5以上</span>
        <span><span style="color: orange; font-weight: bold;">■</span> M3以上</span>
        <span><span style="color: green; font-weight: bold;">■</span> M3未満</span>
    </div>
</div>

<script>
var quakes = {json.dumps(quakes_data)};

window.addEventListener('DOMContentLoaded', (event) => {{
    var mapId = Object.keys(window).find(key => key.startsWith('map_'));
    if (!mapId) return;
    var map = window[mapId];

    quakes.forEach(function(q) {{
        var marker = L.circleMarker([q.lat, q.lon], {{
            radius: Math.max(q.mag * 1.5, 3),
            color: q.color,
            fill: true,
            fillColor: q.color,
            fillOpacity: 0.7
        }}).addTo(map);

        marker.bindTooltip("M " + q.mag);

        marker.on('click', function(e) {{
            var content = `
                <p style="margin: 0 0 6px 0; font-weight: bold; color: #d9534f;">🎯 選択された地震の詳細</p>
                <hr style="margin: 4px 0 8px 0; border: none; border-top: 1px solid #ddd;">
                <p style="margin: 4px 0;"><b>日時:</b> ${{q.time}}</p>
                <p style="margin: 4px 0;"><b>規模:</b> <span style="color: ${{q.color}}; font-weight: bold;">M ${{q.mag}}</span></p>
                <p style="margin: 4px 0;"><b>深さ:</b> ${{q.depth}} km</p>
                <p style="margin: 4px 0; font-size: 11px;"><b>場所:</b><br>${{q.place}}</p>
                <div style="margin-top: 8px; text-align: right;">
                    <button onclick="resetSummary()" style="font-size: 11px; padding: 3px 8px; cursor: pointer; background: #f0f0f0; border: 1px solid #ccc; border-radius: 4px;">全体サマリーに戻る</button>
                </div>
            `;
            document.getElementById('info-content').innerHTML = content;
        }});
    }});
}});

function resetSummary() {{
    document.getElementById('info-content').innerHTML = `
        <p style="margin: 4px 0; color: #666; font-size: 12px;">📌 マップ上のピンをクリックすると、ここに詳細が表示されます。</p>
        <hr style="margin: 8px 0; border: none; border-top: 1px dashed #eee;">
        <p style="margin: 4px 0;">総観測数: <b>{total_count}</b> 件</p>
        <p style="margin: 4px 0;">最大規模: <b style="color: red;">M {max_mag}</b></p>
        <p style="margin: 4px 0; font-size: 11px; color: #333;"><b>最大発生地:</b><br>{max_quake_info}</p>
    `;
}}
</script>
"""

earthquake_map.get_root().html.add_child(Element(

_IncompleteInputError: incomplete input (615183156.py, line 149)